In [3]:
# lib import
import os
import time
import torch
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

# setup
ds_size = 1000000
source_ds_cache_dir = os.path.join(os.getcwd(), "data", "hf-source")

embed_models = [
    "all-MiniLM-L6-v2",
    "sentence-transformers/all-mpnet-base-v2",
    "nomic-ai/nomic-embed-text-v1.5",
]
embed_ds_dirnames = []
for name in embed_models:
    embed_ds_dirnames.append(name.split("/")[-1].replace(".", "-"))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("running on " + device)

running on cuda


# Acquiring the Source Dataset
Using Huggingface's Datasets we can easily load the wikipedia (20231101.en) dataset. Since we are focusing on encyclopedic data, this will serve as the base for testing the dataset generation pipeline.

In [ ]:
source_ds_cache_dir = os.path.join(os.getcwd(), "data", "hf-source")
ds = load_dataset("wikimedia/wikipedia", "20231101.en", cache_dir=source_ds_cache_dir)

In [ ]:
# verification (optional)
print(f"Dataset length: {len(ds['train'])}")
print(ds["train"][0])
print(ds["train"][-1])

In [ ]:
# create subset for dev
ds_shuffled = ds.shuffle(seed=97)
if ds_size > 0:
    ds_subset = ds_shuffled["train"].select(range(ds_size))
else:
    ds_subset = ds_shuffled["train"]

print(f"Test subset length: {len(ds_subset)}")
print(f"Sample entry: {ds_subset[0]['title']}")

# Embedding Generation

In [ ]:
embed_ds = []
print("Preparing texts...")
texts = ds_subset["text"]


for index, model_name in enumerate(embed_models):
    ds_embed_dir = os.path.join(
        os.getcwd(), "data", "ds_embed", embed_ds_dirnames[index], str(ds_size)
    )
    embed_ds.append(ds_subset)

    print("Loading model...")
    model = SentenceTransformer(model_name, trust_remote_code=True, device=device)

    print("Generating embeddings...")
    embedding_start = time.time()

    if model_name == "nomic-ai/nomic-embed-text-v1.5":
        batch_encoding = model.encode(
            texts,
            show_progress_bar=True,
            batch_size=16,
            device=device,
            prompt="clustering: ",
        )
    else:
        batch_encoding = model.encode(
            texts, show_progress_bar=True, batch_size=32, device=device
        )

    embedding_time = time.time() - embedding_start
    print(f"Embedding generation took: {embedding_time:.2f} second(s)")

    embed_ds[index] = embed_ds[index].add_column("embeddings", batch_encoding.tolist())

    print("Saving dataset with embeddings...")
    embed_ds[index].save_to_disk(ds_embed_dir)

# Processing


In [4]:
from numpy import ndarray
from datasets import Dataset, DatasetDict
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA


def prep(data: Dataset | DatasetDict) -> ndarray:
    prep_start = time.time()

    # L2-normalization (to euclidean instead of cosine)
    embeddings_array = np.asarray(data, dtype=np.float32)
    embeddings_norm = normalize(embeddings_array, norm="l2", axis=1)

    prep_time = time.time() - prep_start
    print(f"Data preparation took: {prep_time:.2f} second(s)")

    return embeddings_norm


def prep_pca(data: Dataset | DatasetDict) -> ndarray:
    prep_start = time.time()

    # L2-normalization (to euclidean instead of cosine)
    embeddings_array = np.asarray(data, dtype=np.float32)
    embeddings_norm = normalize(embeddings_array, norm="l2", axis=1)

    # PCA red to 50 dims: De-noising and speedup
    pca = PCA(n_components=50, random_state=97, svd_solver="auto", whiten=False)
    data = pca.fit_transform(embeddings_norm)

    prep_time = time.time() - prep_start
    print(f"Data preparation took: {prep_time:.2f} second(s)")

    return data

In [5]:
from sklearn.preprocessing import MinMaxScaler


def normalize_coords(embed_reduced, target_range=(-1, 1)):
    scaler = MinMaxScaler(feature_range=target_range)
    return scaler.fit_transform(embed_reduced)

# Pipeline A (UMAP)
Based on the standard `UMAP` package.

In [6]:
import umap
from datasets import load_from_disk

for dirname in embed_ds_dirnames:
    ds_embed_dir = os.path.join(os.getcwd(), "data", "ds_embed", dirname, str(ds_size))
    ds_pos_dir = os.path.join(
        os.getcwd(), "data", "ds_pos", dirname, "umap", str(ds_size)
    )
    ds_pos_dir_pca = os.path.join(
        os.getcwd(), "data", "ds_pos", dirname, "umap-pca", str(ds_size)
    )

    ds_embed = load_from_disk(ds_embed_dir)
    print(f"Loaded dataset with {len(ds_embed)} entries")

    # collect embeddings
    embeddings_array = prep(ds_embed["embeddings"])
    embeddings_array_pca = prep_pca(ds_embed["embeddings"])
    print(f"Raw BERT Embeddings shape: {embeddings_array.shape}")
    print(f"PCA pre-processed Embeddings shape: {embeddings_array_pca.shape}")

    # apply UMAP reduction to full-length BERT embeddings
    reducer = umap.UMAP(n_components=2, random_state=97)
    print("Applying UMAP reduction to raw BERT embeddings...")
    reduction_start = time.time()
    embed_reduced = reducer.fit_transform(embeddings_array)
    print(f"2D embeddings shape: {embed_reduced.shape}")
    reduction_time = time.time() - reduction_start
    print(f"UMAP dimensionality reduction took: {reduction_time:.2f} second(s)")

    # apply normalization
    embed_reduced = normalize_coords(embed_reduced)

    # add to ds
    ds_pos = ds_embed.add_column("x", embed_reduced[:, 0].tolist())
    ds_pos = ds_pos.add_column("y", embed_reduced[:, 1].tolist())
    print(f"Final dataset columns: {ds_pos.column_names}")

    # save ds with positions
    ds_pos.save_to_disk(ds_pos_dir)
    print("Dataset with UMAP positions saved successfully!")

    # apply UMAP reduction to pre-processed PCA output
    reducer_pca = umap.UMAP(n_components=2, random_state=97, init="pca")
    print("Applying UMAP reduction to reduced PCA output...")
    reduction_start = time.time()
    embed_reduced_pca = reducer_pca.fit_transform(embeddings_array_pca)
    print(f"2D embeddings shape: {embed_reduced_pca.shape}")
    reduction_time = time.time() - reduction_start
    print(f"UMAP dimensionality reduction took: {reduction_time:.2f} second(s)")

    # apply normalization
    embed_reduced_pca = normalize_coords(embed_reduced_pca)

    # add to ds
    ds_pos_pca = ds_embed.add_column("x", embed_reduced_pca[:, 0].tolist())
    ds_pos_pca = ds_pos_pca.add_column("y", embed_reduced_pca[:, 1].tolist())
    print(f"Final dataset columns: {ds_pos_pca.column_names}")

    # save ds with positions
    ds_pos_pca.save_to_disk(ds_pos_dir_pca)
    print("Dataset with UMAP positions saved successfully!")

Loaded dataset with 1000000 entries
Data preparation took: 12.55 second(s)
Data preparation took: 18.62 second(s)
Raw BERT Embeddings shape: (1000000, 384)
PCA pre-processed Embeddings shape: (1000000, 384)
Applying UMAP reduction to raw BERT embeddings...


c:\Users\ctechadmin\Documents\Philip\acme-paper\src\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


2D embeddings shape: (1000000, 2)
UMAP dimensionality reduction took: 1607.26 second(s)
Final dataset columns: ['id', 'url', 'title', 'text', 'embeddings', 'x', 'y']


Saving the dataset (0/13 shards):   0%|          | 0/1000000 [00:00<?, ? examples/s]

Dataset with UMAP positions saved successfully!
Applying UMAP reduction to reduced PCA output...


...

Dataset with UMAP positions saved successfully!
Applying UMAP reduction to reduced PCA output...


c:\Users\ctechadmin\Documents\Philip\acme-paper\src\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


2D embeddings shape: (1000000, 2)
UMAP dimensionality reduction took: 996.59 second(s)
Final dataset columns: ['id', 'url', 'title', 'text', 'embeddings', 'x', 'y']


Saving the dataset (0/19 shards):   0%|          | 0/1000000 [00:00<?, ? examples/s]

Dataset with UMAP positions saved successfully!


# Pipeline B (t-SNE)
Based on the `openTSNE` implementation of `t-SNE`.

In [7]:
from openTSNE import TSNE
from datasets import load_from_disk

for dirname in embed_ds_dirnames:
    ds_embed_dir = os.path.join(os.getcwd(), "data", "ds_embed", dirname, str(ds_size))
    ds_pos_dir = os.path.join(
        os.getcwd(), "data", "ds_pos", dirname, "tsne", str(ds_size)
    )

    # load dataset with embeddings (reuse from UMAP pipeline)
    ds_embed = load_from_disk(ds_embed_dir)
    print(f"Loaded dataset with {len(ds_embed)} entries")

    # collect embeddings
    embeddings_array = prep_pca(ds_embed["embeddings"])
    print(f"Embeddings shape: {embeddings_array.shape}")

    # apply t-SNE reduction using OpenTSNE
    tsne = TSNE(
        n_components=2,
        random_state=97,
        initialization="pca",
        n_jobs=-1,  # use all available cores
    )
    print("Applying t-SNE reduction...")
    reduction_start = time.time()
    embed_reduced_tsne = tsne.fit(embeddings_array)
    print(f"2D t-SNE embeddings shape: {embed_reduced_tsne.shape}")
    reduction_time = time.time() - reduction_start
    print(f"tSNE dimensionality reduction took: {reduction_time:.2f} second(s)")

    # normalize coords
    embed_reduced_tsne = normalize_coords(embed_reduced_tsne)

    # add to dataset
    ds_pos_tsne = ds_embed.add_column("x", embed_reduced_tsne[:, 0].tolist())
    ds_pos_tsne = ds_pos_tsne.add_column("y", embed_reduced_tsne[:, 1].tolist())
    print(f"Final dataset columns: {ds_pos_tsne.column_names}")

    # save dataset with t-SNE positions
    ds_pos_tsne.save_to_disk(ds_pos_dir)
    print("Dataset with t-SNE positions saved successfully!")

Loaded dataset with 1000000 entries
Data preparation took: 17.23 second(s)
Embeddings shape: (1000000, 50)
Applying t-SNE reduction...
2D t-SNE embeddings shape: (1000000, 2)
tSNE dimensionality reduction took: 1657.48 second(s)
Final dataset columns: ['id', 'url', 'title', 'text', 'embeddings', 'x', 'y']


Saving the dataset (0/13 shards):   0%|          | 0/1000000 [00:00<?, ? examples/s]

Dataset with t-SNE positions saved successfully!


Loading dataset from disk:   0%|          | 0/19 [00:00<?, ?it/s]

Loaded dataset with 1000000 entries
Data preparation took: 36.77 second(s)
Embeddings shape: (1000000, 50)
Applying t-SNE reduction...
2D t-SNE embeddings shape: (1000000, 2)
tSNE dimensionality reduction took: 1512.38 second(s)
Final dataset columns: ['id', 'url', 'title', 'text', 'embeddings', 'x', 'y']


...

Dataset with t-SNE positions saved successfully!


Loading dataset from disk:   0%|          | 0/19 [00:00<?, ?it/s]

Loaded dataset with 1000000 entries
Data preparation took: 36.14 second(s)
Embeddings shape: (1000000, 50)
Applying t-SNE reduction...
2D t-SNE embeddings shape: (1000000, 2)
tSNE dimensionality reduction took: 1365.83 second(s)
Final dataset columns: ['id', 'url', 'title', 'text', 'embeddings', 'x', 'y']


Saving the dataset (0/19 shards):   0%|          | 0/1000000 [00:00<?, ? examples/s]

Dataset with t-SNE positions saved successfully!


# Huggingface Upload

This optional part can be uncommented to upload the 1M dataset. Additional auth configuration and a dataset repository change are necessary.

In [8]:
dirs_train = {
    os.path.join(
        "data", "ds_pos", "all-MiniLM-L6-v2", "tsne", "1000000"
    ): "all_MiniLM_L6_v2_tsne",
    os.path.join(
        "data", "ds_pos", "all-MiniLM-L6-v2", "umap", "1000000"
    ): "all_MiniLM_L6_v2_umap",
    os.path.join(
        "data", "ds_pos", "all-MiniLM-L6-v2", "umap-pca", "1000000"
    ): "all_MiniLM_L6_v2_umap_pca",
    os.path.join(
        "data", "ds_pos", "all-mpnet-base-v2", "tsne", "1000000"
    ): "all_mpnet_base_v2_tsne",
    os.path.join(
        "data", "ds_pos", "all-mpnet-base-v2", "umap", "1000000"
    ): "all_mpnet_base_v2_umap",
    os.path.join(
        "data", "ds_pos", "all-mpnet-base-v2", "umap-pca", "1000000"
    ): "all_mpnet_base_v2_umap_pca",
    os.path.join(
        "data", "ds_pos", "nomic-embed-text-v1-5", "tsne", "1000000"
    ): "nomic_embed_text_v1_5_tsne",
    os.path.join(
        "data", "ds_pos", "nomic-embed-text-v1-5", "umap", "1000000"
    ): "nomic_embed_text_v1_5_umap",
    os.path.join(
        "data", "ds_pos", "nomic-embed-text-v1-5", "umap-pca", "1000000"
    ): "nomic_embed_text_v1_5_umap_pca",
}

for path, config_name in dirs_train.items():
    if not os.path.exists(path):
        print("dataset: [", config_name, "] not found. Skipping...")
        continue
    ds = load_from_disk(path)

    # legacy block (clearing raw token column which is no longer part of the dataset)
    if "token_ids" in ds.column_names:
        ds = ds.remove_columns("token_ids")

    ds.push_to_hub("whatphiliptrains/wikipos", config_name=config_name)


Uploading the dataset shards:   0%|          | 0/13 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/77 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/77 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/77 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/77 [00:00<?, ?ba/s]

...

Creating parquet from Arrow format:   0%|          | 0/53 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/53 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/53 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/53 [00:00<?, ?ba/s]

README.md: 0.00B [00:00, ?B/s]